# 03 — Tools, Tool Schemas & Runtime Context

## Learning requirements
- viết tool với `@tool`;
- thiết kế tên/description/input schema rõ để model chọn đúng;
- phân biệt tool read-only và side-effect;
- hiểu runtime context, state và store ở mức kiến trúc;
- trả structured tool result khi có thể.

**Tool description là context cho model.** Tool mơ hồ -> tool selection kém.

In [ ]:
from langchain.tools import tool

@tool
def calculate_total(price: float, quantity: int) -> dict:
    """Calculate order total from unit price and positive quantity."""
    if price < 0 or quantity <= 0:
        raise ValueError("price must be >= 0 and quantity must be > 0")
    return {"price": price, "quantity": quantity, "total": price * quantity}

print(calculate_total.name)
print(calculate_total.description)
print(calculate_total.args)

In [ ]:
# Bind tools directly to a model to inspect raw tool calls.
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))
from src.providers import get_chat_model

model = get_chat_model()
model_with_tools = model.bind_tools([calculate_total])
ai = model_with_tools.invoke("I buy 4 items at 12.5 each. Calculate the total.")
print("tool_calls =", ai.tool_calls)

## Context scopes

Giữ mental model sau:

| Scope | Ý nghĩa | Ví dụ |
|---|---|---|
| runtime context | immutable invocation/request context | authenticated `user_id`, tenant |
| state | thread/workflow state | messages, current interview step |
| store | cross-thread persisted knowledge | user preferences, project memory |

Không nhét mọi thứ vào prompt string.

In [ ]:
# Tool design exercise — intentionally simple local database.
USERS = {
    "u1": {"name": "Anh", "role": "developer"},
    "u2": {"name": "Timmy", "role": "developer"},
}

@tool
def get_user(user_id: str) -> dict:
    """Read one user profile by exact internal user ID. This tool does not modify data."""
    if user_id not in USERS:
        return {"found": False, "user_id": user_id}
    return {"found": True, "user_id": user_id, **USERS[user_id]}

@tool
def save_note(user_id: str, note: str) -> dict:
    """Persist a user note. This has a side effect and should require authorization in production."""
    # Demo only. Do not use global dict as production persistence.
    USERS.setdefault(user_id, {})["note"] = note
    return {"saved": True, "user_id": user_id}

## Exercise — Tool catalog

Viết thêm:
- `search_project_docs(query)`
- `get_project(project_id)`
- `score_candidate(criteria, evidence)`

Với mỗi tool ghi:
- read/write;
- required permissions;
- expected latency;
- possible failure;
- whether human approval is needed.

## Required output
Một tool catalog tối thiểu 5 tools và test cho:
- valid call,
- invalid args,
- not-found,
- unauthorized side-effect (thiết kế expected behavior).

## Done criteria
Bạn không coi tool chỉ là “Python function”; bạn coi nó là capability boundary có schema + permission + side effects.